# N-Terminal Similarity Analysis — TA Proteins

This notebook explores sequence similarity in the **N-terminal region** of
tail-anchored (TA) proteins using the results from `IDR/idr_analysis.py`.

## What this notebook does

1. **Load** the TA protein IDR results
2. **Extract** N-terminal windows (up to 30 residues before the TMD)
3. **Sequence logo** — information-content logo of the right-aligned windows
4. **Pairwise identity heatmap** — all-vs-all % identity to spot sub-groups
5. **CIDER metrics** — FCR, NCPR, κ, Ω distributions to characterise IDR chemistry
6. **MEME motif discovery** *(optional)* — submit windows to the EBI MEME REST API

### Dependencies

```
pip install logomaker matplotlib pandas numpy requests
```

All are listed in `IDR/requirements.txt`.

In [ ]:
import sys, os

# Make IDR module importable when running from the notebooks/ directory
sys.path.insert(0, os.path.join(os.getcwd(), "..", "IDR"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from idr_analysis import fetch_sequence
from nterm_analysis import (
    extract_windows_from_df,
    right_align_sequences,
    plot_sequence_logo,
    pairwise_identity_matrix,
    plot_identity_heatmap,
    build_cider_dataframe,
    format_fasta,
    submit_meme_ebi,
    poll_meme_ebi,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

## 1 — Load TA protein data

In [ ]:
# Adjust path if running from a different directory
TA_RESULTS = os.path.join("..", "IDR", "ta_idr_results.csv")

ta_df = pd.read_csv(TA_RESULTS)
print(f"Loaded {len(ta_df)} TA proteins")
ta_df[["Entry", "Entry.Name", "Length", "Transmembrane", "cterm_distance"]].head()

## 2 — Download sequences and extract N-terminal windows

Sequences are fetched from the UniProt REST API (one request per protein).
Re-run once and cache to avoid repeated downloads.

In [ ]:
SEQUENCE_CACHE = os.path.join("..", "IDR", "ta_sequences.csv")
WINDOW_SIZE = 30  # residues before TMD to examine

# ── Load or download sequences ────────────────────────────────────────────────
if os.path.exists(SEQUENCE_CACHE):
    seq_df = pd.read_csv(SEQUENCE_CACHE)
    sequences = dict(zip(seq_df["Entry"], seq_df["sequence"]))
    print(f"Loaded {len(sequences)} cached sequences")
else:
    print("Fetching sequences from UniProt (this may take a few minutes)...")
    sequences = {}
    for acc in ta_df["Entry"]:
        seq = fetch_sequence(acc)
        if seq:
            sequences[acc] = seq
    pd.DataFrame(
        [{"Entry": k, "sequence": v} for k, v in sequences.items()]
    ).to_csv(SEQUENCE_CACHE, index=False)
    print(f"Downloaded and cached {len(sequences)} sequences")

# ── Extract windows ───────────────────────────────────────────────────────────
windows = extract_windows_from_df(ta_df, sequences, window_size=WINDOW_SIZE)
print(f"Extracted {len(windows)} N-terminal windows (≤{WINDOW_SIZE} residues before TMD)")

In [ ]:
# Quick look at window lengths
lengths = pd.Series([len(v) for v in windows.values()], name="window_length")
print(lengths.describe())
lengths.hist(bins=20, edgecolor="k")
plt.xlabel("Window length (residues)")
plt.ylabel("Number of proteins")
plt.title("N-terminal window lengths")
plt.tight_layout()
plt.show()

## 3 — Sequence logo

Windows are **right-aligned** so position 0 (rightmost) corresponds to the
residue immediately before the TMD in every protein.  The logo is in
*information content* (bits) — taller letters indicate stronger conservation.

In [ ]:
window_list = list(windows.values())
aligned = right_align_sequences(window_list)

fig, ax = plt.subplots(figsize=(14, 3))
plot_sequence_logo(
    aligned,
    ax=ax,
    title="N-terminal window — right-aligned to TMD boundary",
)

# Label x-axis: distance from TMD (negative = upstream)
n_pos = len(aligned[0])
ax.set_xticks(range(0, n_pos, 5))
ax.set_xticklabels([f"{-(n_pos - 1 - i)}" for i in range(0, n_pos, 5)])
ax.set_xlabel("Position relative to TMD start (0 = TMD boundary)")

plt.tight_layout()
plt.show()

## 4 — Pairwise identity heatmap

An all-vs-all % identity matrix of the N-terminal windows reveals whether
there are **subgroups** of similar sequences hiding in the TA dataset.
If clear clusters appear, consider running motif discovery (Section 6)
separately on each cluster.

In [ ]:
# Use only proteins that have a window (some may lack a parsed TMD)
window_accs  = [acc for acc in ta_df["Entry"] if acc in windows]
window_seqs  = [windows[acc] for acc in window_accs]

# Gene names for tick labels (fall back to accession)
acc_to_gene = dict(zip(ta_df["Entry"], ta_df.get("Gene.Names", ta_df["Entry"])))
labels = [
    str(acc_to_gene.get(acc, acc)).split()[0] if pd.notna(acc_to_gene.get(acc)) else acc
    for acc in window_accs
]

pct_mat = pairwise_identity_matrix(window_seqs)
print(f"Identity matrix: {pct_mat.shape[0]} × {pct_mat.shape[1]}")
print(f"Mean off-diagonal identity: {pct_mat[pct_mat < 100].mean():.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
plot_identity_heatmap(
    pct_mat, labels, ax=ax,
    title="Pairwise % identity of N-terminal windows (right-aligned, 30 residues)",
)
plt.tight_layout()
plt.show()

## 5 — CIDER-style IDR metrics

Four complementary metrics characterise the charge composition and patterning
of each N-terminal window:

| Metric | Meaning | Range |
|--------|---------|-------|
| **FCR** | Fraction of charged residues (K, R, D, E) | [0, 1] |
| **NCPR** | Net charge per residue (positive = cationic) | [−1, 1] |
| **κ** | Charge asymmetry (0 = well-mixed, 1 = segregated) | [0, 1] |
| **Ω** | Clustering of charged + Pro residues (1 = clustered) | [0, 1] |

Reference: Das & Pappu (2013) *PNAS* 110:13392; Holehouse *et al.* (2017).

In [ ]:
cider_df = build_cider_dataframe(window_accs, windows)

# Add gene names for display
cider_df["gene"] = cider_df["Entry"].map(acc_to_gene).fillna(cider_df["Entry"])

print("CIDER metrics summary:")
print(cider_df[["fcr", "ncpr", "kappa", "omega"]].describe().round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
metric_labels = {
    "fcr":   "FCR — Fraction Charged Residues",
    "ncpr":  "NCPR — Net Charge Per Residue",
    "kappa": "κ — Charge Asymmetry",
    "omega": "Ω — Charge+Pro Clustering",
}
for ax, (metric, label) in zip(axes.flat, metric_labels.items()):
    data = cider_df[metric].dropna()
    ax.hist(data, bins=20, edgecolor="k", color="steelblue", alpha=0.8)
    ax.axvline(data.median(), color="tomato", linestyle="--", label=f"median={data.median():.2f}")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

fig.suptitle("CIDER metrics for TA N-terminal windows", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# FCR vs NCPR scatter: Das-Pappu phase diagram style
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    cider_df["fcr"], cider_df["ncpr"],
    c=cider_df["kappa"], cmap="plasma", s=20, alpha=0.7
)
plt.colorbar(sc, ax=ax, label="κ (charge asymmetry)")
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("FCR")
ax.set_ylabel("NCPR")
ax.set_title("Das-Pappu phase diagram (N-terminal windows)")
plt.tight_layout()
plt.show()

## 6 — MEME motif discovery via EBI REST API *(optional)*

De-novo motif discovery on the N-terminal windows using MEME.  
**Requirements:** valid e-mail address and internet access to `ebi.ac.uk`.

The cell below is **not executed by default** (`raise` guards it).  Remove the
`raise` and set your e-mail address to run it.

In [ ]:
raise RuntimeError("Set your e-mail below and remove this line to run MEME.")

EMAIL = "your.email@institution.ac.uk"  # <-- set this

# Build a FASTA string from the N-terminal windows
fasta_text = format_fasta(windows)
print(f"Submitting {len(windows)} sequences to EBI MEME…")

job_id = submit_meme_ebi(
    fasta_text, EMAIL,
    nmotifs=5,   # number of motifs to find
    minw=4,      # minimum motif width
    maxw=15,     # maximum motif width
)
print(f"Job submitted: {job_id}")

print("Polling for results (may take several minutes)…")
meme_output = poll_meme_ebi(job_id, poll_interval=30, max_wait=900)
print(meme_output[:2000])  # print first 2000 chars

## Summary

| Analysis | What to look for |
|----------|------------------|
| Sequence logo | High-information positions near the TMD indicate conserved residues |
| Pairwise identity heatmap | Bright clusters = sub-groups; run MEME per-cluster for sharper motifs |
| FCR distribution | Wide spread → heterogeneous charge content across TA proteins |
| NCPR distribution | Symmetric around 0 → balanced charge; skewed → net bias |
| κ distribution | High κ → charge-segregated IDRs (tend to collapse); low κ → mixed (tend to extend) |
| Ω distribution | High Ω → charge/Pro residues cluster; may indicate functional short motifs |
| Das-Pappu scatter | Reveals whether TA N-termini occupy distinct IDR phase-diagram regions |